# 06 - CamemBERT face aux modèles de référence

**Question** : un modèle de langage français, affiné sur nos données, fait-il mieux que TF-IDF + régression logistique pour deviner NOVA et le Nutri-Score à partir du texte ? Et le gain vaut-il son coût ?

**Protocole**
- Deux tâches « texte seul » (scénarios A de l'étape 6) : NOVA depuis le nom et les catégories, Nutri-Score depuis le texte complet.
- Même jeu de test que l'étape 6 (découpage figé), vérifié produit par produit.
- **Test de McNemar** : sur le même jeu de test, il ne regarde que les produits où les deux modèles sont en désaccord, et dit si l'écart peut être dû au hasard.
- **Règle de décision** pour l'application : CamemBERT n'est retenu que s'il gagne au moins 0,02 de F1 macro **et** que l'écart est significatif. Sinon, la régression logistique est conservée : elle tourne sans GPU, en quelques millisecondes, et ses décisions sont explicables.

**Prérequis** : les fichiers renvoyés par Colab, placés dans `reports/models/` (métriques `.json`) et `data/processed/predictions/` (prédictions `.parquet`).

In [9]:
import pandas as pd

from nutrilys import config
from nutrilys.db import products_connection
from nutrilys.models import compare
from nutrilys.models.dataset import TASKS
from nutrilys.models.evaluate import confusion_shares
from nutrilys.viz import charts
from nutrilys.viz.fmt import fmt_num, fmt_pct
from nutrilys.viz.theme import SERIES, save_figure, use_theme

use_theme()
con = products_connection()
metrics = compare.all_metrics()
tasks = [
    t for t in config.TRANSFORMER_TASKS if t in set(metrics.query("model == 'camembert'")["task"])
]
if not tasks:
    raise FileNotFoundError(
        "Aucun résultat CamemBERT trouvé dans reports/models/ : voir le notebook Colab."
    )
findings = []
print("Tâches avec un résultat CamemBERT :", ", ".join(tasks))

Tâches avec un résultat CamemBERT : nova_name, nutriscore_text


## 1. Scores sur le jeu de test

In [10]:
view = metrics[metrics["task"].isin(tasks)].copy()
view["task_label"] = view["task"].map({k: t.description for k, t in TASKS.items()})
view["model_label"] = view["model"].map(compare.MODEL_LABELS)
colors = {"lr": SERIES[0], "xgb": SERIES[1], "camembert": SERIES[2]}
colors = {k: v for k, v in colors.items() if k in set(view["model"])}

best_gain = view.pivot(index="task", columns="model", values="f1_macro").pipe(
    lambda p: p["camembert"] - p["lr"]
)
lead = best_gain.idxmax()
short = {"nova_name": "NOVA depuis le nom", "nutriscore_text": "Nutri-Score depuis le texte"}
gain_txt = f"{best_gain[lead]:+.3f}".replace(".", ",")
fig = charts.grouped_bars(
    view,
    "task_label",
    "model",
    "f1_macro",
    colors,
    series_labels=compare.MODEL_LABELS,
    title=f"{short[lead]} : CamemBERT fait {gain_txt} de F1 macro face à la régression logistique",
    subtitle="F1 macro sur le même jeu de test, tâches « texte seul »",
    value_label="F1 macro",
    value_range=(0, 1.08),
    value_format=".3f",
)
save_figure(fig, "transformer_f1")
fig.show()

In [11]:
table = view[
    [
        "task_label",
        "model_label",
        "f1_macro",
        "accuracy",
        "within_one",
        "n_train",
        "fit_seconds",
        "predict_ms_per_1000",
    ]
].copy()
table.columns = [
    "Tâche",
    "Modèle",
    "F1 macro",
    "Accuracy",
    "À une classe près",
    "Produits d'entraînement",
    "Entraînement (s)",
    "Prédiction (ms / 1000)",
]
table.style.format(
    {
        "F1 macro": "{:.3f}",
        "Accuracy": "{:.3f}",
        "À une classe près": "{:.3f}",
        "Produits d'entraînement": "{:,.0f}",
        "Entraînement (s)": "{:,.0f}",
        "Prédiction (ms / 1000)": "{:,.0f}",
    }
).hide(axis="index")

Tâche,Modèle,F1 macro,Accuracy,À une classe près,Produits d'entraînement,Entraînement (s),Prédiction (ms / 1000)
NOVA à partir du nom et des catégories,Régression logistique,0.833,0.825,0.949,"223,666",67,70
Nutri-Score à partir du texte,Régression logistique,0.721,0.735,0.942,"190,549",191,200
NOVA à partir du nom et des catégories,CamemBERT (fine-tuning),0.826,0.817,0.949,"223,666",877,538
Nutri-Score à partir du texte,CamemBERT (fine-tuning),0.701,0.715,0.945,"99,999","1,384","1,978"


**Lire les temps avec prudence** : CamemBERT est entraîné et évalué sur le GPU de Colab, les modèles de référence sur le processeur d'un ordinateur portable. Sur une même machine sans GPU, l'écart de temps de prédiction serait bien plus grand encore.

## 2. L'écart est-il réel ? Test de McNemar

In [12]:
rows = []
for task in tasks:
    lr = compare.predictions(task, "lr")
    tr = compare.predictions(task, "camembert")
    result = compare.mcnemar_test(lr, tr)
    model, reason = compare.choose_model(metrics, task, result["p_value"])
    rows.append({"task": task, **result, "choice": model, "reason": reason})
tests = pd.DataFrame(rows)

display_tests = tests.assign(
    Tâche=tests["task"].map({k: t.description for k, t in TASKS.items()}),
    p=tests["p_value"].map(lambda p: "< 0,001" if p < 0.001 else fmt_num(p, 3)),
)[["Tâche", "only_a_correct", "only_b_correct", "accuracy_gap", "p", "choice"]]
display_tests.columns = [
    "Tâche",
    "Seule la rég. logistique a raison",
    "Seul CamemBERT a raison",
    "Écart d'accuracy",
    "p (McNemar)",
    "Modèle retenu pour l'app",
]
display_tests.style.format({"Écart d'accuracy": "{:+.3f}"}).hide(axis="index")

Tâche,Seule la rég. logistique a raison,Seul CamemBERT a raison,Écart d'accuracy,p (McNemar),Modèle retenu pour l'app
NOVA à partir du nom et des catégories,2781,2418,-0.008,"< 0,001",lr
Nutri-Score à partir du texte,4042,3246,-0.019,"< 0,001",lr


In [13]:
for _, row in tests.iterrows():
    label = TASKS[row["task"]].description
    print(f"{label} : {row['reason']}")
    findings.append(f"{label} : {row['reason']}")

NOVA à partir du nom et des catégories : Écart de F1 macro de -0,007, significatif mais inférieur au seuil de +0,02 : la régression logistique, rapide, explicable et sans GPU, est conservée.
Nutri-Score à partir du texte : Écart de F1 macro de -0,020, significatif mais inférieur au seuil de +0,02 : la régression logistique, rapide, explicable et sans GPU, est conservée.


## 3. Où CamemBERT fait-il la différence ?

Matrice de confusion de CamemBERT sur NOVA depuis le nom, à comparer à celle de l'étape 6.

In [14]:
if "nova_name" in tasks:
    tr = compare.predictions("nova_name", "camembert")
    labels = [str(c) for c in config.NOVA_GROUPS]
    shares = confusion_shares(tr["y_true"].astype(int) - 1, tr["y_pred"].astype(int) - 1, labels)
    fig = charts.confusion_heatmap(
        shares,
        title=f"CamemBERT reconnaît {fmt_pct(shares.loc['4', '4'])} des produits "
        "ultra-transformés depuis leur nom",
        subtitle="Matrice de confusion, NOVA depuis le nom et les catégories, "
        "% de chaque groupe réel",
        x_title="Groupe NOVA prédit",
        y_title="Groupe NOVA réel",
    )
    save_figure(fig, "nova_name_camembert_confusion")
    fig.show()

Exemples de produits où **seul CamemBERT** a raison : ce sont typiquement des noms où le sens compte plus que les mots isolés (ordre des mots, formulations rares, synonymes).

In [15]:
task = tasks[0]
merged = compare.predictions(task, "lr").merge(
    compare.predictions(task, "camembert"), on="code", suffixes=("_lr", "_tr")
)
only_tr = merged[
    (merged["y_pred_tr"] == merged["y_true_tr"]) & (merged["y_pred_lr"] != merged["y_true_lr"])
]
con.register("only_tr", only_tr.sort_values("confidence_tr", ascending=False).head(15))
con.execute("""
    SELECT p.product_name AS "Produit", o.y_true_tr AS "Réel",
           o.y_pred_lr AS "Rég. logistique", o.y_pred_tr AS "CamemBERT"
    FROM only_tr o JOIN products p USING (code)
""").df()

,Produit,Réel,Rég. logistique,CamemBERT
0,CHAI,4,1,4
1,Jus de Bissap Mytille,4,1,4
2,Chouquettes,4,3,4
3,Lingots Nougats,4,1,4
4,Miettes extras morue salées sans peau sans arête,4,3,4
5,Bicarbonate Alimentaire,2,1,2
6,Spatzle,4,3,4
7,l'artisan tartineur,4,1,4
8,Dryck Blabar (bosbessendrank) Fles,4,3,4
9,Boissons au bissap,4,3,4


## 4. Synthèse

In [16]:
for line in findings:
    print("-", line)

- NOVA à partir du nom et des catégories : Écart de F1 macro de -0,007, significatif mais inférieur au seuil de +0,02 : la régression logistique, rapide, explicable et sans GPU, est conservée.
- Nutri-Score à partir du texte : Écart de F1 macro de -0,020, significatif mais inférieur au seuil de +0,02 : la régression logistique, rapide, explicable et sans GPU, est conservée.


## 5. Limites

- **Un seul entraînement** : les scores d'un réseau de neurones varient un peu d'un entraînement à l'autre (initialisation, ordre des exemples). Un écart de quelques millièmes n'est pas interprétable sans plusieurs répétitions.
- **Budget limité** : 2 époques sur le GPU gratuit de Colab, longueur de texte tronquée. Un entraînement plus long ou CamemBERT complet pourraient gagner un peu.
- **Comparaison de coûts asymétrique** : GPU pour CamemBERT, processeur pour les modèles de référence.
- **Même cible imparfaite** : comme à l'étape 6, les modèles apprennent les règles d'Open Food Facts, pas une vérité mesurée.